# Legal-Clause GPT: training on Colab

Full fine-tuning of `gpt2` and `distilgpt2` on LEDGAR (100 clause types) with the Hugging Face Trainer.

**Runtime → Change runtime type → T4 GPU**, then run the cells top to bottom (about 1.5 hours in all: ~51 min for gpt2, ~34 min for distilgpt2).
The fine-tuned models go to Google Drive. The small outputs (logits, logs, summaries) are downloaded as
`legal-clause-gpt-outputs.zip`: unzip that into the project folder on your machine, and all evaluation runs locally from there.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the code
Upload `dist/legal-clause-gpt.zip` (made locally with `python scripts/make_colab_zip.py`) when asked.
Once the project is on GitHub, set `REPO_URL` and the cell clones it instead.

In [ ]:
import os, zipfile
REPO_URL = ""   # e.g. "https://github.com/Sky00raider/tinyverse-projects.git" once pushed

if not os.path.exists("/content/legal-clause-gpt"):
    if REPO_URL:
        !git clone -q {REPO_URL} /content/tinyverse
        !ln -s /content/tinyverse/legal-clause-gpt /content/legal-clause-gpt
    else:
        from google.colab import files
        uploaded = files.upload()          # pick legal-clause-gpt.zip
        zipfile.ZipFile(next(iter(uploaded))).extractall("/content")
%cd /content/legal-clause-gpt

In [ ]:
!pip install -q "transformers>=5.0" "accelerate>=1.0" "datasets>=3.0"
import transformers, torch; print("transformers", transformers.__version__, "| torch", torch.__version__)

In [ ]:
!python -m unittest discover -s tests

## 2. Data: download LEDGAR and look at it

In [ ]:
!python scripts/download_data.py
!python scripts/data_stats.py

Check the **"fits uncut in max_len"** line above. 256 tokens fits 85% of clauses, 384 fits 95%: we use 384.
Batches are grouped by length, so the longer limit only costs time on the few long clauses.

## 3. Google Drive for the fine-tuned models (~0.3 GB / 0.5 GB)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/legal-clause-gpt"
!mkdir -p {DRIVE}/models

## 4. Smoke test (~1 min)
30 steps on 2,000 clauses to check the whole pipeline on the GPU. Note the **it/s** in the progress bar:
a full gpt2 run is `3 epochs × 59,080 / 16 = 11,079` steps.

In [ ]:
!python -m src.train --base distilgpt2 --train-size 2000 --eval-size 200 --max-steps 30 --predict-limit 300 --run-name smoke --model-dir /content/smoke_model

## 5. Full runs

In [ ]:
TRAIN_SIZE = 60000   # above 59,080 = the whole deduplicated train split
MAX_LEN = 384

In [ ]:
# main model (51 min on a T4)
!python -m src.train --base gpt2 --train-size {TRAIN_SIZE} --max-len {MAX_LEN} --model-dir {DRIVE}/models/gpt2_n{TRAIN_SIZE//1000}k
!rsync -a --exclude smoke --exclude model outputs/ {DRIVE}/outputs/    # backup in case the session drops

In [ ]:
# comparison model (34 min on a T4)
!python -m src.train --base distilgpt2 --train-size {TRAIN_SIZE} --max-len {MAX_LEN} --model-dir {DRIVE}/models/distilgpt2_n{TRAIN_SIZE//1000}k
!rsync -a --exclude smoke --exclude model outputs/ {DRIVE}/outputs/

## 6. Download the outputs (logits, logs, summaries; no model weights)

In [ ]:
!rm -rf outputs/runs/smoke
!zip -r -q /content/legal-clause-gpt-outputs.zip outputs -x "outputs/runs/*/model/*"
!cat outputs/runs/*/summary.json | grep -E '"run"|accuracy|seconds'
from google.colab import files
files.download("/content/legal-clause-gpt-outputs.zip")